# The Knots You Cannot Read

The recovered flag is **SIGMOID_PROVENANCE**. The payload is in `embedding.weight[128, :18]`, at absolute model byte offsets **17012–17083 inclusive**. Valid raw ASCII bytes range from 0 to 127, so this row is never selected by the specified input pipeline.

These 18 weights are exact multiples of 1/1024. Decode each as `round(1024 * weight + 128)` and interpret the integers as ASCII. The model file contains no literal flag bytes.

Run this notebook from its directory. It needs Python and NumPy; no PyTorch, pretrained replacement model or external data is used.

In [1]:
import json,re,struct,sys
from pathlib import Path
import numpy as np

def load(path):
 raw=Path(path).read_bytes()
 n=struct.unpack('<Q',raw[:8])[0]
 header=json.loads(raw[8:8+n])
 tensors={}
 for name,entry in header.items():
  if name=='__metadata__':continue
  start,end=entry['data_offsets'];count=int(np.prod(entry['shape']))
  if entry['dtype']!='F32' or end-start!=4*count or 8+n+end>len(raw):raise ValueError('Invalid F32 tensor')
  tensors[name]=np.frombuffer(raw,dtype='<f4',offset=8+n+start,count=count).reshape(entry['shape'])
 return raw,header,tensors,8+n

def extract(model):
 raw,header,tensors,base=model
 embedding=tensors['embedding.weight'];width=embedding.shape[1]
 values=embedding[128:].ravel().astype(np.float64)*1024+128
 mask=np.isfinite(values)&(values==np.rint(values))&(values>=32)&(values<=126)
 bounds=np.flatnonzero(np.diff(np.r_[False,mask,False])).reshape(-1,2)
 found=[]
 for start,end in bounds:
  text=bytes(values[start:end].astype(np.uint8)).decode('ascii')
  if re.fullmatch(r'SIGMOID_[A-Z]+',text):
   index=128*width+int(start);row,column=divmod(index,width)
   found.append({'flag':text,'tensor':'embedding.weight','row':row,'column':column,'length':int(end-start),'byte_offset':base+header['embedding.weight']['data_offsets'][0]+4*index,'byte_length':4*int(end-start),'scale':1024,'offset':128,'ascii_codes':values[start:end].astype(int).tolist(),'raw_flag_present':text.encode() in raw})
 if len(found)!=1:raise ValueError('Expected one encoded flag')
 return found[0]

model=load('model.safetensors')
payload=extract(model)
print(json.dumps(payload,indent=2))


{
  "flag": "SIGMOID_PROVENANCE",
  "tensor": "embedding.weight",
  "row": 128,
  "column": 0,
  "length": 18,
  "byte_offset": 17012,
  "byte_length": 72,
  "scale": 1024,
  "offset": 128,
  "ascii_codes": [
    83,
    73,
    71,
    77,
    79,
    73,
    68,
    95,
    80,
    82,
    79,
    86,
    69,
    78,
    65,
    78,
    67,
    69
  ],
  "raw_flag_present": false
}


## Verify the classifier and the unreachable storage

The tensor shapes support a 256-byte embedding, mean pooling over each 64-byte record and a three-output linear classifier. The output-to-register order is inferred from the labelled corpus because the weights contain no label metadata. This reconstruction correctly classifies **2290/2400 records (95.4167%)**; this is a measurement on the supplied corpus, not a separate held-out evaluation.

The audit replaces all embedding rows 128–255 with random values and recomputes every logit. Every logit remains exactly equal, with maximum difference **0.0**. More generally, any valid ASCII input looks up only unchanged rows 0–127, so these unused weights cannot affect its result.

In [2]:
from audit import audit
report=audit(Path('.'))
print(json.dumps(report,indent=2))


{
  "extraction": {
    "flag": "SIGMOID_PROVENANCE",
    "tensor": "embedding.weight",
    "row": 128,
    "column": 0,
    "length": 18,
    "byte_offset": 17012,
    "byte_length": 72,
    "scale": 1024,
    "offset": 128,
    "ascii_codes": [
      83,
      73,
      71,
      77,
      79,
      73,
      68,
      95,
      80,
      82,
      79,
      86,
      69,
      78,
      65,
      78,
      67,
      69
    ],
    "raw_flag_present": false
  },
  "model_sha256": "8e5710a90f735d0331555d99da515703924a1db05afa11dfc8a61a7ab96696e4",
  "corpus_sha256": "4b01efd4aacce26d14b1f9f8c4d8d1fd9a6b76b1f57c3fbf35ce06d2283cfb6d",
  "tensor_shapes": {
    "classifier.bias": [
      3
    ],
    "classifier.weight": [
      3,
      32
    ],
    "embedding.weight": [
      256,
      32
    ]
  },
  "records": 2400,
  "record_length": 64,
  "largest_observed_byte": 121,
  "output_labels": [
    "antisuyu",
    "chinchaysuyu",
    "qullasuyu"
  ],
  "class_counts": {
    "antisuyu": 8